# 1 — From stored numbers to ECG voltages

**Learning objective:** explain how a native ECG file becomes a twelve-channel
voltage array, and reproduce the calibration arithmetic without treating ADC counts
as millivolts. Then understand why lead order is part of the signal contract.

Our path is **choose a public example → inspect structural metadata → decode →
calibrate → reorder → compare with WFDB**. There is no diagnosis prediction or training.
The bundled public files are never edited. All real data are from the official
PhysioNet/CinC Challenge 2021 v1.0.3 release; see [data attribution](../data/ATTRIBUTION.md).

The implementation we investigate is [`signal.py`](../src/ecg_waveform_lab/signal.py):
`read_native`, `calibrate` and `reorder_leads`. Its contract is twelve channels in
rows, samples in columns, at the original sampling frequency. This is an
educational MAT-file reader, not support for every WFDB storage format.

## Setup: use this checkout's installed package

Run `uv sync --locked` and launch `uv run --frozen jupyter lab notebooks` from the
repository root. Choose its Python 3 kernel and run this notebook in order.
Each tutorial works independently. The installed package locates the bundled
sample relative to the checkout; there are no machine-specific paths or downloads.
The environment versions below make your numerical session inspectable.

In [ ]:
import os
from importlib.metadata import version

from ecg_waveform_lab import project_root

root = project_root()
os.environ.setdefault("MPLCONFIGDIR", str(root / ".cache/matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from ecg_waveform_lab import (
    CANONICAL_LEADS,
    calibrate,
    collate_windows,
    load_catalog,
    make_window,
    prepare_ecg,
    read_native,
    reorder_leads,
    resample_signal,
    verify_sample,
    WaveformDataset,
)

display(pd.Series({
    package: version(package)
    for package in ("numpy", "scipy", "wfdb", "torch", "matplotlib")
}, name="Installed version"))

**What we learned:** the notebook imports the installed local implementation and
uses the declared numerical environment. It does not need state from another notebook.

## 1.1 Choose a record using engineering properties

The small catalog records only public file provenance and structural properties.
For this first lesson, choose a 500 Hz, ten-second PTB-XL example. Changing
`example_index` among the available candidates changes the public waveform, not
any cohort or label policy. We do not select using diagnoses.

In [ ]:
catalog = load_catalog(root)
candidates = [record for record in catalog if record.cohort == "ptb-xl"]
example_index = 0
record = candidates[example_index]
print("Public record:", record.record_id)
print("Source cohort:", record.cohort)
print("Selection reason:", record.selection_reason)

WFDB parses the official header. Display only acquisition fields: channel names,
gains, baselines and units. `gain` means **ADC counts per physical unit**; `baseline`
is the ADC value representing zero in that physical scale. It is not the average
of this recording and does not remove physiological baseline wander.

In [ ]:
import wfdb

header = wfdb.rdheader(str(record.header_path.with_suffix("")))
header_table = pd.DataFrame({
    "lead": header.sig_name,
    "gain_counts_per_unit": header.adc_gain,
    "baseline_counts": header.baseline,
    "physical_unit": header.units,
})
display(header_table)
print("Native rate (Hz):", header.fs)
print("Samples per channel:", header.sig_len)
print("Nominal duration (s):", header.sig_len / header.fs)

**What we learned:** storage metadata define how to interpret each channel's numbers.
Header demographics and diagnosis comments are not inputs to our signal operations.

## 1.2 Decode the MAT container

The Challenge stores a `val` matrix in MATLAB v4 files. `read_native` delegates
container decoding to SciPy, checks dimensions and finite integer ADC values, and
uses the parsed calibration. The result exposes both counts and millivolts so we
can inspect the transformation. No lead-order permutation or resampling has happened.

The WFDB `16+24` format describes signed 16-bit samples after a byte offset in the
original file. A MAT reader already handles its container; applying that offset
again to `val` would discard valid data. Transposing an unexpected matrix by guess
would also hide an error, so the loader requires the declared shape.

In [ ]:
native = read_native(record.header_path)
print("Stored ADC shape:", native.digital.shape)
print("Stored ADC dtype:", native.digital.dtype)
print("Calibrated shape:", native.millivolts.shape)
print("Calibration dtype:", native.millivolts.dtype)
print("Native lead order:", native.lead_names)
print("First eight ADC counts in channel 0:", native.digital[0, :8])

**What we learned:** the decoded matrix has a precise channel/time orientation.
The float64 calibrated array remains at the native rate; decoding is not resampling.

## 1.3 Work through ADC calibration

For channel $c$ and sample $t$,

$$v_{c,t}^{\mathrm{mV}} = \frac{d_{c,t}-b_c}{g_c}\,s_c.$$

Here $d$ is the digital count, $b$ the baseline in counts, $g$ the gain in counts
per header unit, and $s$ the conversion from that unit to mV (1 for mV, 0.001 for
µV, 1000 for V). The explicit baseline, when present, takes precedence over the
ADC-zero field according to WFDB's header convention. Missing gain/units are
rejected by this teaching reader rather than silently assigning default calibration.

First, an **illustrative arithmetic example**: counts `[100, 1100, -900]`, baseline
100 and gain 1000 counts/mV should produce `[0, 1, -1]` mV. Try changing the gain
and predict whether the voltage range grows or shrinks. This is not a stored ECG.

In [ ]:
illustrative_counts = np.array([100.0, 1100.0, -900.0])
illustrative_baseline = 100.0
illustrative_gain = 1000.0

counts_relative_to_zero = illustrative_counts - illustrative_baseline
illustrative_millivolts = counts_relative_to_zero / illustrative_gain
print("Counts relative to zero:", counts_relative_to_zero)
print("Illustrative voltage (mV):", illustrative_millivolts)

Calibration differs by lead, so a `(12,)` vector must apply along the channel axis
of a `(12, samples)` matrix. NumPy aligns broadcasting dimensions from the right.
Adding a singleton time axis gives `(12, 1)`: each channel's calibration is reused
at every sample. A bare `(12,)` vector would align with time, which is wrong here.

In [ ]:
gain_column = native.gain[:, None]
baseline_column = native.baseline[:, None]
print("Digital array:", native.digital.shape)
print("Gain column:", gain_column.shape)
print("Baseline column:", baseline_column.shape)

calibrated_again = calibrate(
    native.digital, native.gain, native.baseline, native.units
)
np.testing.assert_allclose(calibrated_again, native.millivolts)
print("The public calibration function reproduces the decoded voltages.")

Inspect a few paired values and a channel's voltage range. A positive/negative
voltage is relative to the stated reference. These numbers alone cannot establish
correct electrode placement or clinical quality, and there is no amplitude clipping.

In [ ]:
lead_name = "II"
channel = native.lead_names.index(lead_name)
display(pd.DataFrame({
    "ADC_count": native.digital[channel, :8],
    "calibrated_mV": native.millivolts[channel, :8],
}))
print("Lead II voltage range (mV):", (
    native.millivolts[channel].min(), native.millivolts[channel].max()
))

**What we learned:** ADC calibration is a channel-specific physical conversion,
not normalisation, baseline removal or a learned transformation. Broadcasting makes
that same conversion apply efficiently to every sample.

## 1.4 Plot counts and millivolts on the same time axis

Try `view_seconds = 1.0` or another `lead_name`. The waveform's time structure should
stay the same under an affine calibration, while the vertical scale changes.
At 500 Hz the time step is 2 ms; sample zero is at 0 seconds. These are scientific
plots, not a calibrated clinical ECG printout with paper speed and gain settings.

In [ ]:
view_seconds = 2.0
channel = native.lead_names.index(lead_name)
visible_samples = min(
    round(view_seconds * native.sample_rate_hz),
    native.digital.shape[1],
)
time_seconds = np.arange(visible_samples) / native.sample_rate_hz

figure, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True, constrained_layout=True)
axes[0].plot(time_seconds, native.digital[channel, :visible_samples])
axes[0].set(
    ylabel="ADC counts",
    title=f"Public example, lead {lead_name}: stored values",
)
axes[1].plot(time_seconds, native.millivolts[channel, :visible_samples])
axes[1].set(
    xlabel="Time (s)",
    ylabel="Voltage (mV)",
    title="Calibrated physical values",
)
for axis in axes:
    axis.grid(alpha=0.2)
plt.show()

**What we learned:** decoding and calibration preserve the waveform's timing.
An amplitude axis without units can hide a thousandfold scale error.

## 1.5 Treat lead order as part of the interface

A downstream array consumer sees channel indices, not electrode names. A correctly
scaled but permuted signal can therefore be misinterpreted. The package requires
each canonical lead exactly once and rejects missing, duplicated or unknown names.
It never reconstructs a lead or silently fills a missing channel with zeros.

The bundled example may already be canonical. To make reordering visible, create a
clearly identified **in-memory permutation demonstration**. Both values and names
must move together. No original file or clinical observation is altered.

In [ ]:
permutation_seed = 17
permutation = np.random.default_rng(permutation_seed).permutation(12)
permuted_values = native.millivolts[permutation].copy()
permuted_names = tuple(native.lead_names[index] for index in permutation)
canonical_values = reorder_leads(permuted_values, permuted_names)

display(pd.DataFrame({
    "incoming_channel": np.arange(12),
    "incoming_lead": permuted_names,
    "canonical_channel_for_this_lead": [
        CANONICAL_LEADS.index(name) for name in permuted_names
    ],
}))
np.testing.assert_allclose(
    canonical_values, reorder_leads(native.millivolts, native.lead_names)
)
print("Output lead order:", CANONICAL_LEADS)

**What we learned:** reorder values using verified names; do not merely relabel the
rows. Try a different permutation seed and explain why the restored signal is unchanged.

## 1.6 Compare with a reference decoder

WFDB can read the same native files directly into physical values. Here its
physical units are millivolts (the release uses both `mV` and `mv`; check
them before comparing). SciPy's MAT decoding and the
package's explicit calibration should agree with WFDB. They share the same header
information, so agreement does **not** independently prove the header was calibrated
correctly at acquisition or that the ECG is clinically interpretable.

In [ ]:
reference = wfdb.rdrecord(str(record.header_path.with_suffix("")), physical=True)
print("Header unit spelling:", sorted(set(reference.units)))
assert all(unit in ("mV", "mv") for unit in reference.units)
reference_millivolts = reference.p_signal.T
maximum_difference = np.max(np.abs(reference_millivolts - native.millivolts))
print("Reference shape:", reference_millivolts.shape)
print("Maximum absolute difference (mV):", maximum_difference)
np.testing.assert_allclose(reference_millivolts, native.millivolts, atol=1e-10, rtol=0)

**What we learned:** the public example decodes consistently through two container
reading paths, with explicit shape, units and channel order. We have not changed
the sample rate or constructed a fixed window yet.

**Explain back:** why is an ADC count not a voltage? Why is subtracting the baseline
not the same as removing baseline wander? Why is `(12, 1)` the useful calibration
shape? What does decoder agreement fail to establish?

Continue to [resampling, windows and batches](02_resampling_windows_and_batches.ipynb).
The exact APIs and deliberate limits are documented in [the signal contract](../docs/SIGNAL_CONTRACT.md).